In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers
from tensorflow.keras.datasets import cifar10

import numpy as np
import matplotlib.pyplot as plt

from keras import backend as K
print(K.backend())

2026-04-09 11:08:17.186383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


tensorflow


In [2]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
# load the cifar-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.2,
    random_state=123,
    stratify=y_train
)

y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)
y_test = to_categorical(y_test, 10)

x_train, x_test = x_train / 255, x_test / 255

print(x_train.shape)
print(y_train.shape)

/home/juhana/anaconda3_v2/envs/keras/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


(40000, 32, 32, 3)
(40000, 10)


In [3]:
import tensorflow as tf

# Train dataset
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))

# Validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))

# Test dataset
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))

batch_size = 32

train_dataset = train_dataset.shuffle(buffer_size=1000).batch(batch_size)
val_dataset = val_dataset.batch(batch_size)
test_dataset = test_dataset.batch(batch_size)

I0000 00:00:1775722103.759958   84872 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
2026-04-09 11:08:23.762930: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 983040000 exceeds 10% of free system memory.
2026-04-09 11:08:25.589328: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 983040000 exceeds 10% of free system memory.


In [4]:
from keras.applications import VGG16
from keras import Sequential
from keras import layers

conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

conv_base.trainable = False

In [5]:
import keras_hub

preprocessor = keras_hub.layers.ImageConverter.from_preset(
    "xception_41_imagenet",
    image_size=(32, 32),
)

/home/juhana/anaconda3_v2/envs/keras/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def get_features_and_labels(dataset):
    all_features = []
    all_labels = []
    for images, labels in dataset:
        preprocessed_images = preprocessor(images)
        features = conv_base.predict(preprocessed_images, verbose=0)
        all_features.append(features)
        all_labels.append(labels)
    return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(val_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

2026-04-09 11:08:28.704588: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 983040000 exceeds 10% of free system memory.
2026-04-09 11:08:29.723761: I external/local_xla/xla/service/service.cc:163] XLA service 0x7ee4840048d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-09 11:08:29.723790: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-04-09 11:08:29.734775: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-04-09 11:08:29.805293: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92000
2026-04-09 11:08:30.622217: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 662.00MiB with freed_by_count=0. The cal

In [ ]:
print(train_features.shape)

In [ ]:
num_classes = 10

model = Sequential([
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction.keras",
        save_best_only=True,
        monitor="val_loss",
    )
]

history = model.fit(
    train_features,
    train_labels,
    epochs=10,
    validation_data=(val_features, val_labels),
    callbacks=callbacks,
)